# Taller Pandas - Estadísticas de Jugadores 2024/2025

Dataset: Football Players Stats 2024-2025 (Kaggle)

**Preguntas:**
1. ¿Cuál es el promedio de goles y asistencias según la posición?
2. ¿Qué liga tiene los jugadores más agresivos (tarjetas por 90 min)?
3. ¿Cuáles son los 10 jugadores más eficientes en goles por 90 min (con al menos 900 minutos jugados)?

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('players_data-2024_2025.csv')
df.shape

In [ ]:
df.head(3)

In [ ]:
# son muchas columnas, me quedo solo con las que voy a usar
cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'MP', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'CrdY', 'CrdR', 'xG', 'xAG']
df = df[cols]
df.head()

In [ ]:
df.isnull().sum()

## Limpieza

In [ ]:
df = df.dropna(subset=['Nation', 'Age'])
df['Age'] = df['Age'].astype(int)
df.shape

In [ ]:
# algunos jugadores tienen posicion doble tipo 'FW,MF', me quedo con la principal
df['Pos'].value_counts().head(10)

In [ ]:
df['Pos_Principal'] = df['Pos'].apply(lambda x: x.split(',')[0])
df['Pos_Principal'].value_counts()

## Nuevas variables

In [ ]:
df['Gls_por_90'] = np.where(df['90s'] > 0, (df['Gls'] / df['90s']).round(2), 0)
df['Total_Tarjetas'] = df['CrdY'] + df['CrdR']
df['Tarjetas_por_90'] = np.where(df['90s'] > 0, (df['Total_Tarjetas'] / df['90s']).round(3), 0)

df[['Player', 'Gls', 'Gls_por_90', 'Total_Tarjetas', 'Tarjetas_por_90']].head()

---
## Pregunta 1 - Promedio de goles y asistencias por posicion

In [ ]:
df.groupby('Pos_Principal')[['Gls', 'Ast', 'G+A']].mean().round(2).sort_values('G+A', ascending=False)

In [ ]:
# cuantos jugadores hay por posicion
df.groupby('Pos_Principal')['Player'].count()

**Conclusion:** Los delanteros (FW) tienen los promedios mas altos tanto en goles como asistencias, lo cual tiene todo el sentido. Los mediocampistas tambien aportan bastante. Porteros y defensas casi no aparecen en estadisticas ofensivas.

---
## Pregunta 2 - Que liga es la mas agresiva?

In [ ]:
# filtro jugadores con al menos 5 partidos para no distorsionar
df_5mp = df[df['MP'] >= 5].copy()
print(f'Jugadores con 5+ partidos: {len(df_5mp)}')

In [ ]:
liga_stats = df_5mp.groupby('Comp').agg(
    Prom_CrdY=('CrdY', 'mean'),
    Prom_CrdR=('CrdR', 'mean'),
    Tarjetas_por_90=('Tarjetas_por_90', 'mean'),
    N_jugadores=('Player', 'count')
).round(3).sort_values('Tarjetas_por_90', ascending=False)

liga_stats

In [ ]:
print(f"Liga mas agresiva: {liga_stats['Tarjetas_por_90'].idxmax()}")

**Conclusion:** La Liga española lidera en agresividad con el mayor promedio de tarjetas por 90 min. La Ligue 1 resulta ser la mas disciplinada de las 5 grandes ligas, algo que no era obvio.

---
## Pregunta 3 - Top 10 goleadores mas eficientes (min. 900 min)

In [ ]:
# solo delanteros y medios con minutos suficientes
df_ofensivos = df[(df['Min'] >= 900) & (df['Pos_Principal'].isin(['FW', 'MF']))].copy()
print(f'Jugadores en el filtro: {len(df_ofensivos)}')

In [ ]:
top10 = df_ofensivos.sort_values('Gls_por_90', ascending=False).head(10)
top10[['Player', 'Squad', 'Comp', 'Age', 'Min', 'Gls', '90s', 'Gls_por_90']]

In [ ]:
prom_top10 = top10['Gls_por_90'].mean().round(3)
prom_general = df_ofensivos['Gls_por_90'].mean().round(3)
print(f'Top 10 promedio: {prom_top10} | General: {prom_general} | Ratio: {round(prom_top10/prom_general, 1)}x')

**Conclusion:** El Top 10 es casi 5 veces mas eficiente que el promedio. Sorloth lidera con 1.15 goles por 90 min, por encima incluso de Mbappe y Lewandowski. Tiene sentido filtrar por minutos minimos, sin ese filtro aparecen jugadores que jugaron 10 minutos y metieron un gol, lo cual no es representativo.